# CHIRPS

In [ ]:
# make sure we're running from scripts/download so `import common` resolves
import os, sys

NOTEBOOK_DIR = os.path.abspath(".")
if os.path.basename(NOTEBOOK_DIR) != "download":
    NOTEBOOK_DIR = os.path.join(NOTEBOOK_DIR, "scripts", "download")
os.chdir(NOTEBOOK_DIR)
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)
print("cwd:", os.getcwd())

In [ ]:
import gzip
import shutil

from common import RAW_DIR, download_file, ensure_dir, get_logger

log = get_logger("chirps")

BASE_URL = "https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_monthly/tifs"
OUT_DIR = RAW_DIR / "chirps"

YEAR_START = 2020
YEAR_END = 2024  # inclusive


def main() -> None:
    ensure_dir(OUT_DIR)
    for year in range(YEAR_START, YEAR_END + 1):
        for month in range(1, 13):
            fname_gz = f"chirps-v2.0.{year}.{month:02d}.tif.gz"
            fname_tif = fname_gz[:-3]
            dest_gz = OUT_DIR / fname_gz
            dest_tif = OUT_DIR / fname_tif

            if dest_tif.exists():
                log.info("Already have it, skipping: %s", dest_tif)
                continue

            url = f"{BASE_URL}/{fname_gz}"
            try:
                download_file(url, dest_gz)
            except Exception as e:  # noqa: BLE001
                log.warning("Failed to download %s: %s", url, e)
                continue

            with gzip.open(dest_gz, "rb") as f_in, open(dest_tif, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)
            dest_gz.unlink()

    log.info("Done. Files are in %s", OUT_DIR)

In [ ]:
main()